# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [21]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5004 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [22]:
import functools
from typing import Any

# TODO: Implementacja dekoratora
def show_list_length(func : Any) -> Any:
    @functools.wraps(func)
    def wrapper(*args : Any, **kwargs : Any) -> Any:
        for arg in args:
            if isinstance(arg, (list, set)):
                print(f"Argument: {arg}")
        result = func(*args, **kwargs)
        return result
    return wrapper

# Test:
@show_list_length
def process_data(data_list : list[Any], name: str):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4, 5], "danych")
process_data({1, 2, 3}, "zbioru")
process_data("nie jest listą", "stringiem")

Argument: [1, 2, 3, 4, 5]
Przetwarzanie danych
Argument: {1, 2, 3}
Przetwarzanie zbioru
Przetwarzanie stringiem


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [23]:
import functools
from datetime import datetime
import time

# TODO: Implementacja dekoratora z argumentem
def logger(filename : str) -> Any:
    def decorator(func: Any) -> Any:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            start_time = datetime.now()
            result = func(*args, **kwargs)
            end_time = datetime.now()
            with open(filename, "a") as f:
                f.write(f"{func.__name__} started at {start_time} and ended at {end_time}\n")
            return result
        return wrapper
    return decorator

@logger("log.txt")
def long_task():
    time.sleep(1)
    print("Długie zadanie zakończone.")

long_task()

Długie zadanie zakończone.


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [24]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [25]:
class AccessLogger:
    def __set_name__(self, owner : type, name: str) -> None:
        self.name = name

    def __get__(self, instance : Any, owner: type) -> Any:
        value = instance.__dict__.get(self.name)
        print(f"Dostęp do {self.name}: {value}")
        return value
    
    def __set__(self, instance : Any, value: Any) -> None:
        print(f"Ustawianie {self.name} na {value}")
        instance.__dict__[self.name] = value

class Uzytkownik:
    imie : AccessLogger
    wiek : AccessLogger

    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie: str, wiek: int):
        self.imie = imie
        self.wiek = wiek

u = Uzytkownik("Anna", 30)
print(u.imie)
u.wiek = 31
print(u.wiek)

Ustawianie imie na Anna
Ustawianie wiek na 30
Dostęp do imie: Anna
Anna
Ustawianie wiek na 31
Dostęp do wiek: 31
31


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [26]:
class FibonacciGenerator:
    def __init__(self, limit : int):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
def collatz_generator(n: int):
    while n != 1:
        yield n
        if n % 2 == 0:
            n //= 2
        else:
            n = 3 * n + 1
    yield 1

# Test:
for status in collatz_generator(10):
    print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [33]:
current_user : dict[str, str]

def require_role(role: str) -> Any:
    def decorator(func: Any) -> Any:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            if current_user["role"] != role:
                raise PermissionError(f"Użytkownik {current_user['username']} nie ma uprawnień do wykonania tej operacji.")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("superuser")
def delete_user(username: str):
    print(f"Użytkownik {username} został usunięty.")

@require_role("moderator")
def view_dashboard():
    print("Wyświetlanie panelu administracyjnego.")

@require_role("user")
def view_profile():
    print("Wyświetlanie profilu użytkownika.")

# Testing

current_user = {"username": "admin", "role": "superuser"}
print(f"\nZalogowany jako: {current_user['username']} ({current_user['role']})")

try:
    delete_user("test_user")
except PermissionError as e:
    print(e)
try:
    view_dashboard()
except PermissionError as e:
    print(e)
try:
    view_profile()
except PermissionError as e:
    print(e)

current_user = {"username": "mod", "role": "moderator"}
print(f"\nZalogowany jako: {current_user['username']} ({current_user['role']})")

try:
    delete_user("test_user")
except PermissionError as e:
    print(e)
try:
    view_dashboard()
except PermissionError as e:
    print(e)
try:
    view_profile()
except PermissionError as e:
    print(e)

current_user = {"username": "regular_user", "role": "user"}
print(f"\nZalogowany jako: {current_user['username']} ({current_user['role']})")

try:
    delete_user("test_user")
except PermissionError as e:
    print(e)
try:
    view_dashboard()
except PermissionError as e:
    print(e)
try:
    view_profile()
except PermissionError as e:
    print(e)


Zalogowany jako: admin (superuser)
Użytkownik test_user został usunięty.
Użytkownik admin nie ma uprawnień do wykonania tej operacji.
Użytkownik admin nie ma uprawnień do wykonania tej operacji.

Zalogowany jako: mod (moderator)
Użytkownik mod nie ma uprawnień do wykonania tej operacji.
Wyświetlanie panelu administracyjnego.
Użytkownik mod nie ma uprawnień do wykonania tej operacji.

Zalogowany jako: regular_user (user)
Użytkownik regular_user nie ma uprawnień do wykonania tej operacji.
Użytkownik regular_user nie ma uprawnień do wykonania tej operacji.
Wyświetlanie profilu użytkownika.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [36]:
class Typed:
    expected_type : type

    def __init__(self, expected_type: type):
        self.expected_type = expected_type

    def __set_name__(self, owner : type, name: str):
        self.name = name

    def __set__(self, instance : object, value : object):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"Oczekiwano typu {self.expected_type.__name__} dla atrybutu {self.name} (otrzymano {type(value).__name__})")
        instance.__dict__[self.name] = value

    def __get__(self, instance : object, owner : type) -> object:
        return instance.__dict__.get(self.name)
    
class Person:
    name : Typed = Typed(str)
    age : Typed = Typed(int)

    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

try:
    p1 = Person("Alice", 30)
    print(f"Utworzono osobę: {p1.name}, {p1.age} lat")
except TypeError as e:
    print(e)

try:
    p2 = Person("Bob", "thirty")  # Powinno rzucić błąd
    print(f"Utworzono osobę: {p2.name}, {p2.age} lat")
except TypeError as e:
    print(e)

Utworzono osobę: Alice, 30 lat
Oczekiwano typu int dla atrybutu age (otrzymano str)


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [41]:
class PrimeGenerator:
    def __init__(self):
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        self.current += 1
        while True:
            if self.is_prime(self.current):
                return self.current
            self.current += 1

    @staticmethod
    def is_prime(n: int) -> bool:
        if n <= 1:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True

def prime_generator(limit: int = 10):
    generator : PrimeGenerator = PrimeGenerator()
    for prime in generator:
        if prime > limit:
            break
        yield prime

# Test:
for prime in (p for p in prime_generator(1000) if p % 10 == 7):
    print(prime)


7
17
37
47
67
97
107
127
137
157
167
197
227
257
277
307
317
337
347
367
397
457
467
487
547
557
577
587
607
617
647
677
727
757
787
797
827
857
877
887
907
937
947
967
977
997
